<a href="https://colab.research.google.com/github/lenzip/DataAnalysisSubnuclearPhysics/blob/main/bw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fit of a Breit-Wigner in RooFit
In this notebook we generate a sample of events distributed as a Breit-Wigner and then we fit the parameters of the Breit-Wigner, using RooFit.

> **Technical note**
>
> To run in Google Colab you need to install ROOT. This is achieved in the cell below. The cell prints out the OS on the Colab machine and then picks (in the `wget` command) the ROOT binaries for the same OS. *Note: if the OS for Colab changes you may need to update the `wget` command.*
>
>Then, it loads the library in the current environment.



In [ ]:
try:
  import google.colab
  !lsb_release -a
  !sudo apt install -y libgsl-dev
  !mkdir -p APPS
  #check that you are dowloading the correct build, matching the OS from https://root.cern.ch/download/
  !cd APPS && wget https://root.cern.ch/download/root_v6.36.12.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
  !cd APPS && tar -xf root_v6.36.12.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
  import sys
  sys.path.append("/content/APPS/root/lib")
  import ctypes
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libCore.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libThread.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libImt.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libRIO.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libNet.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libTree.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libMathCore.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libMatrix.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libHist.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libGraf.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libRooFit.so')
  ctypes.cdll.LoadLibrary('/content/APPS/root/lib/libRooFitMore.so')
except ImportError:
  pass

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.5 LTS
Release:	22.04
Codename:	jammy
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgsl-dev is already the newest version (2.7.1+dfsg-3).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
--2026-04-27 13:58:53--  https://root.cern.ch/download/root_v6.36.12.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
Resolving root.cern.ch (root.cern.ch)... 137.138.6.72, 2001:1458:201:8b::100:23b
Connecting to root.cern.ch (root.cern.ch)|137.138.6.72|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 301040008 (287M) [application/x-gzip]
Saving to: ‘root_v6.36.12.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz.2’

root_v6.36.12.Linux 100%[===================>] 287.09M  28.3MB/s    in 11s     

2026-04-27 13:59:04 (27.2 MB/s) - ‘root_v6.36.12.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz.2’ saved [301040008/301040008]


gzip: stdin: unexpected end of file
t

In [ ]:
import ROOT

## Part 1: generate 100 events distributed as a Breit-Wigner
In the following we generate 100 extractions of a random variable distributed according to a Breit-Wigner with mean 90 GeV and width 2.5 GeV. We also want to restrict the range of the random variable to [60-120] GeV.

We first neet a couple of `import`s.

> Note that `from ROOT import *` is not allowed.

In [ ]:
import ROOT
import array
from ROOT.RooFit import Range, Bins

### Define the random variable
We call the random variable `mll`, and we define its range.

In [ ]:
#mll   = ROOT.RooRealVar("mll", "m_{ll}", 60, 120, 'GeV')
mll   = ROOT.RooRealVar(unit='GeV', title="m_{ll}", name="mll", minValue=60, maxValue=120)

### Define the Breit Wigner pdf
Within ROOT, the `RooBreitWigner` type is defined to represent a Breit-Wigner distribution. It is documented [here](https://root.cern.ch/doc/master/classRooBreitWigner.html).

As you can see it takes two parameters, the mean and the width. It also takes as input the random variable.

So we need to define the parameters corresponding to the mean and the width, variables `m0` and `Gamma` in the following, and then call the constructor of the `RooBreitWigner`.


In [ ]:
m0    = ROOT.RooRealVar("m0", "m_{0}", 90, unit='GeV')
Gamma = ROOT.RooRealVar("gamma", "#Gamma", 2.5, unit='GeV')
bw    = ROOT.RooBreitWigner("bw", "bw", mll, m0, Gamma)

### Generate a 100 events dataset
We finally get to generate 100 extractions of the PDF, using method `RooAbsPdf::generate` method.
This is an inherited method for `RooBreitWigner`, it having `RooAbsPdf` as an ancestor in its inheritance graph.

To find the description of the `generate` method (and to find out it even exists), you need to click on "Public member functions inherited from RooAbsPdf" in the [documentation page](https://root.cern.ch/doc/master/classRooBreitWigner.html) of `RooBreitWigner`:
![Description](https://drive.google.com/uc?id=1J77VE8CH5taacTo7fSF240Xqo-MKK-cQ)

The `generate` method returns a `RooDataSet`, i.e. an object containing the several extractions of the random variable. In this simple case, the `RooDataSet` is 1D, but if you have a PDF describing a set of random variables, it will be multidimentional.


In [ ]:
dataset = bw.generate(ROOT.RooArgSet(mll), Name="data", NumEvents=100, Verbose=True)

 --- RooGenContext --- 
Using PDF RooBreitWigner::bw[ x=mll mean=m0 width=gamma ]
Use PDF generator for ()
Use MC sampling generator RooFoamGenerator for (mll)


### Plotting
Plotting RooFit objects has a slightly different logic than in ROOT.

One first needs to define the quantity they want to plot, in this case the random variable represented by the `mll` object.

We do this by calling the method `frame` of `RooRealVar`, documented [here](https://root.cern.ch/doc/master/classRooAbsRealLValue.html#a2908be9b0c0f21a72345e77cf3ddb30d). As you can see, it can take several arguments, including for example a modified range (through the `Range` argument), or the number of bins (through the `Bins` argument).

Then we plot objects on the frame. This includes both the PDF and the dataset.

> **The order matters**: if you plot a PDF first, it will be normalized to 1. If you plot it after a dataset, it will be normalized to the number of events in the dataset.

In [ ]:
%jsroot on
frame = mll.frame(Range=(80,110), Bins=50, Title="m_{ll} spectrum")
dataset.plotOn(frame)
bw.plotOn(frame)

frame.Draw()
ROOT.gPad.Draw()

## Part 2: Fit the parameters of the Breit-Wigner
In this part we will assume that the data we just generated actually come from an experiment, and we want to estimate the parameters of the Breit-Wigner.

### Define the PDF we want to fit and its parameters
We first need to define a new object representing the PDF we want to fit.
This new PDF comes with its own parameters.

The crucial difference with respect to the first time we defined a Breit Wigner is that this time the parameters themselves have a range.
The fit will find the maximum of the likelihood in those ranges.

In [ ]:
m0fit    = ROOT.RooRealVar("m0fit", "m_{0}", 80, 110, unit='GeV')
Gammafit = ROOT.RooRealVar("gammafit", "#Gamma", 0, 10, unit='GeV')
bwfit    = ROOT.RooBreitWigner("bwfit", "bwfit", mll, m0fit, Gammafit)

### Fiting the data
The ML fit to the data is encapsulated in `RooAbsPdf::fitTo`, documented [here](https://root.cern.ch/doc/master/classRooAbsPdf.html#ab0721374836c343a710f5ff92a326ff5), which `RooBreitWigner` inherits.

The call `bwfit.fitTo(dataset, ROOT.RooFit.Minos(m0fit))` below builds the likelihood, creates the NLL, finds it minimum (calling function `MIGRAD` of Minuit), finds the covariance matrix of the parameters according to the RCF inequality (calling function `HESSE` of Minuit) and finally performs the 'graphical method' evaluation of the uncertainty on parameter of interese `m0fit` (using Minuit's `MINOS` function).

In [ ]:
bwfit.fitTo(dataset, ROOT.RooFit.Minos(m0fit))
m0fitRes = m0fit.getVal()
m0fitresDo = m0fit.getVal()+m0fit.getAsymErrorLo()
m0fitresUp = m0fit.getVal()+m0fit.getAsymErrorHi()
print (m0fitRes, m0fitresDo, m0fitresUp)

90.1289830983678 89.9493121727494 90.3094410587426
[#1] INFO:Fitting -- RooAbsPdf::fitTo(bwfit_over_bwfit_Int[mll]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_bwfit_over_bwfit_Int[mll]_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
Minuit2Minimizer: Minimize with max-calls 1000 convergence for edm < 1 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 239.349871200604667
Edm   = 2.95701541730374736e-09
Nfcn  = 22
gammafit	  = 2.43142	 +/-  0.330237	(limited)
m0fit	  = 90.129	 +/-  0.179679	(limited)
******************************************************************************************************
Minuit2Minimizer::GetMinosError - Run MINOS LOWER error for parameter #1 : m0fit using max-calls 1000, tolerance 1
*****************************************************************

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =        239.349877 Edm =   5.770231954e-06 NCalls =      5
Info in <Minuit2>: MnSeedGenerator Initial state  
  Minimum value : 239.349877
  Edm           : 5.770231954e-06
  Internal parameters:	[    -0.5392510899    -0.3307247758]	
  Internal gradient  :	[    0.04361891627    0.03855382517]	
  Internal covariance matrix:
[[    0.011880597              0]
 [              0  0.00032076694]]]
Info in <Minuit2>: VariableMetricBuilder Start iterating until Edm is < 0.001 with call limit = 1000
Info in <Minuit2>: VariableMetricBuilder    0 - FCN =        239.349877 Edm =   5.770231954e-06 NCalls =      5
Info in <Minuit2>: VariableMetricBuilder    1 - FCN =       239.3498712 Edm =   2.878491021e-09 NCalls =     10
Info in <Minuit2>: VariableMetricBuilder After Hessian
Info in <Minuit2>: VariableMetricBuilder    2 - FCN =       239.3498712 Edm =   2.957

### Ploting the result
After the fit each parameter is at its best fit value, i.e. if you call `m0fit.getVal()` it will give you the best fit of `m0fit`. We can plot the best fit PDF exactly like before.

In [ ]:
frame_mll = mll.frame()
dataset.plotOn(frame_mll)
bwfit.plotOn(frame_mll)
frame_mll.Draw()
ROOT.gPad.Draw()

### Let's brake it down into its pieces

The `fitTo` method is very convenient, but it hides a lot of the complexity, which we may want, or need, to expose.

We can in fact manually create the NLL using method `RooAbsPdf::createNLL` method, documented [here](https://root.cern.ch/doc/master/classRooAbsPdf.html#a24b1afec4fd149e08967eac4285800de).

Then, we can manually feed that function to Minuit and call `MIGRAD` to get the best fit.

In [ ]:
nll = bwfit.createNLL(dataset)
ROOT.RooMinimizer(nll).migrad()
minNLL=nll.getVal()

[#1] INFO:Fitting -- RooAbsPdf::fitTo(bwfit_over_bwfit_Int[mll]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_bwfit_over_bwfit_Int[mll]_data) Summation contains a RooNLLVar, using its error level
Minuit2Minimizer: Minimize with max-calls 1000 convergence for edm < 1 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 239.349871197672513
Edm   = 2.38637083782436189e-11
Nfcn  = 24
gammafit	  = 2.43142	 +/-  0.330237	(limited)
m0fit	  = 90.129	 +/-  0.179679	(limited)


Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =       239.3498712 Edm =   2.877358689e-09 NCalls =      5
Info in <Minuit2>: MnSeedGenerator Initial state  
  Minimum value : 239.3498712
  Edm           : 2.877358689e-09
  Internal parameters:	[    -0.5395101993    -0.3307309592]	
  Internal gradient  :	[  0.0001220387625   0.005944950742]	
  Internal covariance matrix:
[[    0.011869694              0]
 [              0  0.00032065285]]]
Info in <Minuit2>: VariableMetricBuilder Start iterating until Edm is < 0.001 with call limit = 1000
Info in <Minuit2>: VariableMetricBuilder    0 - FCN =       239.3498712 Edm =   2.877358689e-09 NCalls =      5
Info in <Minuit2>: VariableMetricBuilder    1 - FCN =       239.3498712 Edm =   1.464956433e-12 NCalls =     10
Info in <Minuit2>: VariableMetricBuilder After Hessian
Info in <Minuit2>: VariableMetricBuilder    2 - FCN =       239.3498712 Edm =   2.38

### Draw the likelihood around its minimum
Now that we have the NLL stored in variable `nll`, we can play with it.
For example, we can draw it around it minimum.

> The `nll` object is a function of parameters `m0fit` and `Gammafit`. We can plot the NLL for any value of the parameters by doing:
> ```python
> m0fit.setVal(<some value>)
> Gammafit.setVal(<some value>)
> nll.getVal() # this will return the value of the NLL at the chosen point in parameter space.
> ```

Let us try and plot the NLL sampling evenly the parameter space in `m0fit` and `Gammafit`.
To do that we build a 2D histogram, in which the x and y represent the tho parmaeters, while the bin content in z is the value of the NLL.

We will take the bin center of the 2D histogtram in x and y as the points of the parameter space where we want to evaluate the NLL.
In z we will report $-2\mathrm{ln}\frac{\mathcal{L(m_0, \Gamma)}}{\mathcal{L(\hat{m_0}, \hat{\Gamma)}}}$.

We show the contours at +1 and +4 in z, and on top we draw lines corresponding to the interval on `m0fit` extracted from MINOS. Notice anything?

In [ ]:
h2d = ROOT.TH2F("2d", "2d", 100, 89., 90.5, 100, 2., 4.)
for i in range(1,h2d.GetXaxis().GetNbins()+1):
  for j in range(1,h2d.GetYaxis().GetNbins()+1):
    m0here    = h2d.GetXaxis().GetBinCenter(i)
    gammaHere = h2d.GetYaxis().GetBinCenter(j)
    m0fit.setVal(m0here)
    Gammafit.setVal(gammaHere)
    h2d.SetBinContent(i, j, 2*(nll.getVal()-minNLL))

levels = ROOT.std.vector('double')()
levels.push_back(1.0)
levels.push_back(4.0)
h2dclone = h2d.Clone()
h2dclone.SetContour(len(levels), levels.data())
h2dclone.GetXaxis().SetTitle("m_{0}")
h2dclone.GetYaxis().SetTitle("#Gamma")
h2dclone.Draw("CONT2 LIST")

line1 = ROOT.TLine(m0fitresDo, 2, m0fitresDo, 4)
line2 = ROOT.TLine(m0fitresUp, 2, m0fitresUp, 4)
line1.Draw("sames")
line2.Draw("sames")
ROOT.gPad.Update()

ROOT.gPad.Draw()

Warning in <TROOT::Append>: Replacing existing TH1: 2d (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: 2d (Potential memory leak).


### Create the profile likelihood in parameter `m0fit`
Now we will show how to create the profile likelihood in parameter `m0fit`.
We remind that, assuming $m_0$ and $\Gamma$ are the mathematical parameter corresponding to variables `m0fit` and `Gammafit`, the profile likelihood in parameter $m_0$ is

$q(m_0)=-2\mathrm{log}\frac{\mathcal{L}(m_0,\hat{\hat{\Gamma}})}{\mathcal{L}(\hat{m_0},\hat{\Gamma})}$

where $\mathcal{L}(m_0,\hat{\hat{\Gamma}})$ is the likelihood evaluated at at the value of $\Gamma$ that maximises it for the chosen value of $m_0$.

To construct $q(m_0)$ point by point we need to perform many fits, one for each value of $m_0$.
This procedure is often referred to as a *likelihood scan*, and we can say that we are *profiling over parameter $\Gamma$*.

This is conveniently achieved with the `createProfile` method of `RooRealVar`.

The several fits are then performed when plotting.

In [ ]:
profile = nll.createProfile(m0fit)
frame1 = m0fit.frame(Bins=20000, Range=(89,90.5), Title="profileLL in mass")
profile.plotOn(frame1)
frame1.GetYaxis().SetRangeUser(0, 2)
frame1.Draw()
line = ROOT.TLine(89, 0.5, 90.5, 0.5)
line.Draw("sames")
print(m0fit.getAsymErrorLo())
line1 = ROOT.TLine(m0fitresDo, 0, m0fitresDo, 2)
line2 = ROOT.TLine(m0fitresUp, 0, m0fitresUp, 2)
line1.Draw("sames")
line2.Draw("sames")
ROOT.gPad.Update()
ROOT.gPad.Draw()


0.0
[#1] INFO:Minimization -- RooProfileLL::evaluate(RooEvaluatorWrapper_Profile[m0fit]) Creating instance of MINUIT
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_bwfit_over_bwfit_Int[mll]_data) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooProfileLL::evaluate(RooEvaluatorWrapper_Profile[m0fit]) determining minimum likelihood for current configurations w.r.t all observable
[#1] INFO:Minimization -- RooProfileLL::evaluate(RooEvaluatorWrapper_Profile[m0fit]) minimum found at (m0fit=90.131)
............................................................................................................................................................................................................................................................................................................................................................................................................................................................................